# 🧠 Nivel 3 — Lógica Difusa: Priorización de Facturas

**Integración al pipeline LangChain:** Este módulo toma las facturas procesadas por el agente (Nivel 2) y les asigna una **prioridad de pago** usando un sistema de inferencia difusa.

## Justificación

En una empresa que recibe múltiples facturas diarias, el humano debe decidir **cuáles pagar primero**. Esta decisión involucra incertidumbre:
- Un monto "alto" no es un umbral exacto — depende del contexto
- La urgencia es gradual, no binaria

La lógica difusa modela esta incertidumbre con **conjuntos difusos** y **reglas IF-THEN**, replicando el razonamiento humano pero de forma automatizada.

## Variables Lingüísticas

| Variable | Tipo | Universo | Conjuntos Difusos |
|---|---|---|---|
| Monto | Entrada | $0 - $500,000 | baja, media, alta |
| Urgencia | Entrada | 0 - 30 días | baja, media, alta |
| Prioridad | Salida | 0 - 100 | baja, media, alta, crítica |

## Método de Desfusificación
Centroide (Center of Gravity)

## 1. Instalación

In [ ]:
!pip install -q scikit-fuzzy numpy matplotlib pandas

## 2. Imports y Configuración

In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime, timedelta

print("✅ Imports cargados")

## 3. Definición de Variables Difusas

In [ ]:
# === VARIABLES DE ENTRADA ===

# Monto de la factura (en pesos colombianos)
monto = ctrl.Antecedent(np.arange(0, 500001, 1000), 'monto')

# Urgencia: días transcurridos desde la emisión de la factura
urgencia = ctrl.Antecedent(np.arange(0, 31, 1), 'urgencia')

# === VARIABLE DE SALIDA ===

# Prioridad de pago (0-100)
prioridad = ctrl.Consequent(np.arange(0, 101, 1), 'prioridad')

print("✅ Variables difusas definidas")
print(f"   Monto: universo [0, 500,000] COP")
print(f"   Urgencia: universo [0, 30] días")
print(f"   Prioridad: universo [0, 100]")

## 4. Funciones de Membresía

In [ ]:
# === FUNCIONES DE MEMBRESÍA: MONTO ===
monto['baja'] = fuzz.trapmf(monto.universe, [0, 0, 30000, 80000])
monto['media'] = fuzz.trimf(monto.universe, [50000, 150000, 300000])
monto['alta'] = fuzz.trapmf(monto.universe, [200000, 350000, 500000, 500000])

# === FUNCIONES DE MEMBRESÍA: URGENCIA (días desde emisión) ===
# Más días = más urgente (la factura está más vencida)
urgencia['baja'] = fuzz.trapmf(urgencia.universe, [0, 0, 3, 8])
urgencia['media'] = fuzz.trimf(urgencia.universe, [5, 12, 20])
urgencia['alta'] = fuzz.trapmf(urgencia.universe, [15, 22, 30, 30])

# === FUNCIONES DE MEMBRESÍA: PRIORIDAD (salida) ===
prioridad['baja'] = fuzz.trapmf(prioridad.universe, [0, 0, 15, 30])
prioridad['media'] = fuzz.trimf(prioridad.universe, [20, 40, 60])
prioridad['alta'] = fuzz.trimf(prioridad.universe, [50, 70, 85])
prioridad['critica'] = fuzz.trapmf(prioridad.universe, [75, 85, 100, 100])

print("✅ Funciones de membresía definidas")

## 5. Visualización de Funciones de Membresía

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9))

# Monto
axes[0].set_title('Funciones de Membresía: MONTO (COP)', fontsize=12, fontweight='bold')
for label in monto.terms:
    axes[0].plot(monto.universe, monto[label].mf, linewidth=2, label=label)
axes[0].legend(loc='upper right')
axes[0].set_xlabel('Monto ($)')
axes[0].set_ylabel('Grado de pertenencia')
axes[0].grid(True, alpha=0.3)

# Urgencia
axes[1].set_title('Funciones de Membresía: URGENCIA (días)', fontsize=12, fontweight='bold')
for label in urgencia.terms:
    axes[1].plot(urgencia.universe, urgencia[label].mf, linewidth=2, label=label)
axes[1].legend(loc='upper right')
axes[1].set_xlabel('Días desde emisión')
axes[1].set_ylabel('Grado de pertenencia')
axes[1].grid(True, alpha=0.3)

# Prioridad
axes[2].set_title('Funciones de Membresía: PRIORIDAD (salida)', fontsize=12, fontweight='bold')
for label in prioridad.terms:
    axes[2].plot(prioridad.universe, prioridad[label].mf, linewidth=2, label=label)
axes[2].legend(loc='upper right')
axes[2].set_xlabel('Prioridad (0-100)')
axes[2].set_ylabel('Grado de pertenencia')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('funciones_membresia.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Gráfica guardada: funciones_membresia.png")

## 6. Reglas IF-THEN (Base de Conocimiento)

In [ ]:
# === REGLAS DE INFERENCIA DIFUSA ===
# Mínimo 5 reglas requeridas — implementamos 7 para mayor cobertura

regla1 = ctrl.Rule(monto['alta'] & urgencia['alta'], prioridad['critica'])
# Si el monto es ALTO y la urgencia es ALTA → prioridad CRÍTICA

regla2 = ctrl.Rule(monto['alta'] & urgencia['media'], prioridad['alta'])
# Si el monto es ALTO y la urgencia es MEDIA → prioridad ALTA

regla3 = ctrl.Rule(monto['media'] & urgencia['alta'], prioridad['alta'])
# Si el monto es MEDIO y la urgencia es ALTA → prioridad ALTA

regla4 = ctrl.Rule(monto['media'] & urgencia['media'], prioridad['media'])
# Si el monto es MEDIO y la urgencia es MEDIA → prioridad MEDIA

regla5 = ctrl.Rule(monto['baja'] & urgencia['alta'], prioridad['media'])
# Si el monto es BAJO y la urgencia es ALTA → prioridad MEDIA

regla6 = ctrl.Rule(monto['baja'] & urgencia['media'], prioridad['baja'])
# Si el monto es BAJO y la urgencia es MEDIA → prioridad BAJA

regla7 = ctrl.Rule(monto['baja'] & urgencia['baja'], prioridad['baja'])
# Si el monto es BAJO y la urgencia es BAJA → prioridad BAJA

print("✅ 7 reglas IF-THEN definidas")
print()
print("Reglas:")
print("  R1: IF monto=alto  AND urgencia=alta  THEN prioridad=crítica")
print("  R2: IF monto=alto  AND urgencia=media THEN prioridad=alta")
print("  R3: IF monto=medio AND urgencia=alta  THEN prioridad=alta")
print("  R4: IF monto=medio AND urgencia=media THEN prioridad=media")
print("  R5: IF monto=bajo  AND urgencia=alta  THEN prioridad=media")
print("  R6: IF monto=bajo  AND urgencia=media THEN prioridad=baja")
print("  R7: IF monto=bajo  AND urgencia=baja  THEN prioridad=baja")

## 7. Sistema de Control Difuso

In [ ]:
# Crear sistema de control con las reglas
sistema_prioridad = ctrl.ControlSystem([
    regla1, regla2, regla3, regla4, regla5, regla6, regla7
])

# Simulador para evaluar facturas
simulador = ctrl.ControlSystemSimulation(sistema_prioridad)

print("✅ Sistema de control difuso creado")
print(f"   Método de desfusificación: Centroide (defuzzify_method='centroid')")
print(f"   Reglas activas: {len(sistema_prioridad.rules)}")

## 8. Función de Evaluación

In [ ]:
def evaluar_prioridad(monto_valor, dias_desde_emision):
    """Evalúa la prioridad de pago de una factura usando lógica difusa.
    
    Args:
        monto_valor: Valor total de la factura en COP
        dias_desde_emision: Días transcurridos desde la fecha de emisión
    
    Returns:
        dict con score numérico y categoría lingüística
    """
    # Limitar a los universos definidos
    monto_valor = max(0, min(500000, monto_valor))
    dias_desde_emision = max(0, min(30, dias_desde_emision))
    
    sim = ctrl.ControlSystemSimulation(sistema_prioridad)
    sim.input['monto'] = monto_valor
    sim.input['urgencia'] = dias_desde_emision
    
    try:
        sim.compute()
        score = sim.output['prioridad']
    except:
        score = 50.0  # Default si no hay regla aplicable
    
    # Clasificar en categoría lingüística
    if score >= 75:
        categoria = "🔴 CRÍTICA"
    elif score >= 55:
        categoria = "🟠 ALTA"
    elif score >= 35:
        categoria = "🟡 MEDIA"
    else:
        categoria = "🟢 BAJA"
    
    return {
        "score": round(score, 2),
        "categoria": categoria
    }

# Test rápido
print("=== Test del sistema difuso ===")
test_cases = [
    (400000, 25),  # Monto alto, muchos días → debería ser CRÍTICA
    (150000, 12),  # Monto medio, urgencia media → MEDIA
    (20000, 2),    # Monto bajo, pocos días → BAJA
    (350000, 5),   # Monto alto, pocos días → MEDIA-ALTA
    (50000, 28),   # Monto bajo, muy vencida → MEDIA
]

for m, d in test_cases:
    result = evaluar_prioridad(m, d)
    print(f"  Monto=${m:,.0f} | Días={d} → Score={result['score']} | {result['categoria']}")

## 9. Integración con Pipeline LangChain

Tomamos los datos de las facturas procesadas por el agente y les aplicamos el sistema de priorización difusa.

In [ ]:
# Cargar datos reales del CSV exportado por el agente (Nivel 2)
import os

CSV_PATH = "registro_facturas.csv"

if os.path.exists(CSV_PATH):
    df_facturas = pd.read_csv(CSV_PATH)
    print(f"✅ CSV cargado: {CSV_PATH}")
    print(f"   Registros: {len(df_facturas)}")
    display(df_facturas)
else:
    print(f"⚠️ No se encontró '{CSV_PATH}'. Asegúrate de ejecutar primero agente_facturas.ipynb")
    print("   Usando datos de ejemplo como fallback...")
    df_facturas = pd.DataFrame([
        {"nit_emisor": "1058816252", "razon_social": "CLAUDIA JHOANA ESCOBAR PINEDA",
         "numero_factura": "QC92", "fecha_emision": "2026-06-06", "valor_total": 13000,
         "descripcion": "CLORO PASTILLA 20 GR", "ruta_pdf": "", "ruta_xml": ""},
        {"nit_emisor": "1097404801", "razon_social": "VALENTINA GARZÓN MARIN",
         "numero_factura": "FE14022", "fecha_emision": "2026-06-06", "valor_total": 250000,
         "descripcion": "Servicios profesionales", "ruta_pdf": "", "ruta_xml": ""},
        {"nit_emisor": "901911083", "razon_social": "EMPRESA XYZ SAS",
         "numero_factura": "LP123", "fecha_emision": "2026-05-20", "valor_total": 450000,
         "descripcion": "Suministros de oficina", "ruta_pdf": "", "ruta_xml": ""}
    ])

print(f"\n📋 Facturas a priorizar: {len(df_facturas)}")

In [ ]:
# Aplicar lógica difusa a cada factura del CSV
hoy = datetime.now()

resultados_priorizados = []

for _, factura in df_facturas.iterrows():
    # Calcular días desde emisión
    try:
        fecha_emision = datetime.strptime(str(factura["fecha_emision"]), "%Y-%m-%d")
        dias = (hoy - fecha_emision).days
    except:
        dias = 0

    # Asegurar que valor_total es numérico
    monto_val = float(factura.get("valor_total", 0) or 0)

    # Evaluar con sistema difuso
    resultado = evaluar_prioridad(monto_val, dias)

    resultados_priorizados.append({
        "nit_emisor": factura.get("nit_emisor", "N/A"),
        "razon_social": factura.get("razon_social", "N/A"),
        "numero_factura": factura.get("numero_factura", "N/A"),
        "fecha_emision": factura.get("fecha_emision", "N/A"),
        "valor_total": monto_val,
        "descripcion": factura.get("descripcion", "N/A"),
        "dias_desde_emision": dias,
        "prioridad_score": resultado["score"],
        "prioridad_categoria": resultado["categoria"]
    })

# Crear DataFrame ordenado por prioridad
df_priorizado = pd.DataFrame(resultados_priorizados)
df_priorizado = df_priorizado.sort_values("prioridad_score", ascending=False).reset_index(drop=True)

print("\n" + "="*70)
print("📊 FACTURAS PRIORIZADAS POR LÓGICA DIFUSA")
print("="*70)
print(f"Fecha actual: {hoy.strftime('%Y-%m-%d')}")
print(f"Método: Inferencia Difusa Mamdani + Desfusificación por Centroide")
print("="*70)

display(df_priorizado[[
    "numero_factura", "razon_social", "valor_total",
    "dias_desde_emision", "prioridad_score", "prioridad_categoria"
]])

## 10. Visualización de Resultados

In [ ]:
# Visualizar la superficie de control 3D
from mpl_toolkits.mplot3d import Axes3D

# Generar malla de puntos
x_monto = np.arange(0, 500001, 10000)
y_urgencia = np.arange(0, 31, 1)
X, Y = np.meshgrid(x_monto, y_urgencia)
Z = np.zeros_like(X, dtype=float)

# Evaluar cada punto
for i in range(len(y_urgencia)):
    for j in range(len(x_monto)):
        result = evaluar_prioridad(X[i, j], Y[i, j])
        Z[i, j] = result["score"]

# Gráfica 3D
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(X, Y, Z, cmap='RdYlGn_r', alpha=0.8)

ax.set_xlabel('Monto (COP)', fontsize=10)
ax.set_ylabel('Días desde emisión', fontsize=10)
ax.set_zlabel('Prioridad', fontsize=10)
ax.set_title('Superficie de Control: Sistema Difuso de Priorización', fontsize=12, fontweight='bold')

# Marcar las facturas reales
for _, row in df_priorizado.iterrows():
    ax.scatter(row['valor_total'], row['dias_desde_emision'], row['prioridad_score'],
              color='blue', s=100, zorder=5, edgecolors='black')

fig.colorbar(surf, shrink=0.5, label='Prioridad')
plt.tight_layout()
plt.savefig('superficie_control.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Gráfica guardada: superficie_control.png")

In [ ]:
# Gráfica de barras de prioridad por factura
fig, ax = plt.subplots(figsize=(10, 5))

colores = []
for score in df_priorizado['prioridad_score']:
    if score >= 75:
        colores.append('#d32f2f')  # Rojo - Crítica
    elif score >= 55:
        colores.append('#f57c00')  # Naranja - Alta
    elif score >= 35:
        colores.append('#fbc02d')  # Amarillo - Media
    else:
        colores.append('#388e3c')  # Verde - Baja

bars = ax.barh(
    df_priorizado['numero_factura'] + ' - ' + df_priorizado['razon_social'].str[:20],
    df_priorizado['prioridad_score'],
    color=colores,
    edgecolor='black',
    linewidth=0.5
)

# Líneas de referencia
ax.axvline(x=75, color='red', linestyle='--', alpha=0.5, label='Umbral Crítica')
ax.axvline(x=55, color='orange', linestyle='--', alpha=0.5, label='Umbral Alta')
ax.axvline(x=35, color='gold', linestyle='--', alpha=0.5, label='Umbral Media')

ax.set_xlabel('Score de Prioridad (0-100)')
ax.set_title('Priorización de Facturas por Lógica Difusa', fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlim(0, 100)
ax.grid(True, alpha=0.3, axis='x')

# Agregar valores en las barras
for bar, score in zip(bars, df_priorizado['prioridad_score']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{score:.1f}', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('priorizacion_facturas.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Gráfica guardada: priorizacion_facturas.png")

## 11. Trazabilidad de una Conclusión

Ejemplo detallado del proceso de inferencia para una factura específica.

In [ ]:
# Trazabilidad completa para la factura de mayor prioridad
factura_ejemplo = df_priorizado.iloc[0]

m_val = factura_ejemplo['valor_total']
d_val = factura_ejemplo['dias_desde_emision']

print("="*60)
print("🔍 TRAZABILIDAD DE INFERENCIA DIFUSA")
print("="*60)
print(f"\nFactura: {factura_ejemplo['numero_factura']} - {factura_ejemplo['razon_social']}")
print(f"Monto: ${m_val:,.0f} COP")
print(f"Días desde emisión: {d_val}")

# Paso 1: Fusificación
print("\n--- Paso 1: FUSIFICACIÓN ---")
print(f"Grados de pertenencia para monto=${m_val:,.0f}:")
for term_name in ['baja', 'media', 'alta']:
    degree = fuzz.interp_membership(monto.universe, monto[term_name].mf, m_val)
    print(f"  μ_monto_{term_name}({m_val}) = {degree:.4f}")

print(f"\nGrados de pertenencia para urgencia={d_val} días:")
for term_name in ['baja', 'media', 'alta']:
    degree = fuzz.interp_membership(urgencia.universe, urgencia[term_name].mf, d_val)
    print(f"  μ_urgencia_{term_name}({d_val}) = {degree:.4f}")

# Paso 2: Evaluación de reglas
print("\n--- Paso 2: EVALUACIÓN DE REGLAS (operador MIN) ---")
reglas_texto = [
    ("R1", "monto=alto & urgencia=alta → crítica", 'alta', 'alta'),
    ("R2", "monto=alto & urgencia=media → alta", 'alta', 'media'),
    ("R3", "monto=medio & urgencia=alta → alta", 'media', 'alta'),
    ("R4", "monto=medio & urgencia=media → media", 'media', 'media'),
    ("R5", "monto=bajo & urgencia=alta → media", 'baja', 'alta'),
    ("R6", "monto=bajo & urgencia=media → baja", 'baja', 'media'),
    ("R7", "monto=bajo & urgencia=baja → baja", 'baja', 'baja'),
]

for nombre, texto, m_term, u_term in reglas_texto:
    m_deg = fuzz.interp_membership(monto.universe, monto[m_term].mf, m_val)
    u_deg = fuzz.interp_membership(urgencia.universe, urgencia[u_term].mf, d_val)
    activation = min(m_deg, u_deg)
    if activation > 0:
        print(f"  {nombre}: {texto}")
        print(f"       MIN({m_deg:.4f}, {u_deg:.4f}) = {activation:.4f} ✓")

# Paso 3: Resultado
print(f"\n--- Paso 3: DESFUSIFICACIÓN (Centroide) ---")
resultado = evaluar_prioridad(m_val, d_val)
print(f"  Score final: {resultado['score']}")
print(f"  Categoría: {resultado['categoria']}")
print(f"\n  Conclusión: La factura {factura_ejemplo['numero_factura']} tiene prioridad")
print(f"  {resultado['categoria']} con un score de {resultado['score']}/100")

## 12. Métricas del Sistema

In [ ]:
print("="*60)
print("📈 MÉTRICAS DEL SISTEMA DE LÓGICA DIFUSA")
print("="*60)
print(f"")
print(f"Variables de entrada: 2 (monto, urgencia)")
print(f"Variable de salida: 1 (prioridad)")
print(f"Conjuntos difusos totales: 10")
print(f"  - Monto: 3 (baja, media, alta)")
print(f"  - Urgencia: 3 (baja, media, alta)")
print(f"  - Prioridad: 4 (baja, media, alta, crítica)")
print(f"Reglas IF-THEN: 7")
print(f"Método de inferencia: Mamdani")
print(f"Operador AND: MIN (mínimo)")
print(f"Método de desfusificación: Centroide")
print(f"")
print(f"--- Resultados sobre {len(df_priorizado)} facturas ---")
print(f"Score promedio: {df_priorizado['prioridad_score'].mean():.2f}")
print(f"Score máximo: {df_priorizado['prioridad_score'].max():.2f}")
print(f"Score mínimo: {df_priorizado['prioridad_score'].min():.2f}")
print(f"")
print(f"Distribución de categorías:")
for cat in ['🔴 CRÍTICA', '🟠 ALTA', '🟡 MEDIA', '🟢 BAJA']:
    count = len(df_priorizado[df_priorizado['prioridad_categoria'] == cat])
    print(f"  {cat}: {count} facturas")

## 13. Análisis Crítico y Mejoras Futuras

### ¿Cómo se integra al pipeline LangChain?
El sistema difuso actúa como un **post-procesador inteligente** que recibe el DataFrame de facturas del agente (Nivel 2) y agrega una columna de prioridad. Esto permite que el usuario final vea las facturas ordenadas por urgencia de pago sin intervención manual.

### ¿Qué mejora cuantificable aporta respecto al Nivel 2?
- **Sin lógica difusa**: el usuario ve las facturas en orden de llegada (FIFO)
- **Con lógica difusa**: las facturas se ordenan por urgencia real, considerando monto + tiempo
- **Mejora**: reducción del riesgo de mora en facturas de alto monto/antigüedad

### Limitaciones identificadas
1. Las funciones de membresía están calibradas manualmente — en producción se ajustarían con datos históricos
2. Solo considera 2 variables de entrada (monto y urgencia); no incluye historial del proveedor
3. El universo de monto está limitado a $500,000 — facturas mayores se saturan

### Propuesta de mejora futura
- Agregar variable "confiabilidad del proveedor" basada en historial de pagos
- Usar aprendizaje (ANFIS) para ajustar automáticamente las funciones de membresía
- Integrar con el agente como una Tool más que el LLM pueda invocar